<a href="https://colab.research.google.com/github/zhilyaevaviktorija/machine_learning/blob/main/homeworks/Homework_4_%22w2v_hw_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [ ]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [ ]:
w2v_model_1 = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model_1.key_to_index)}")
print(f"Размерность векторов: {w2v_model_1.vector_size}")

Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

In [ ]:
import gensim.downloader as api

# проверка имеющихся моделей и датасетов
info_datasets = api.info()
print(info_datasets)

{'corpora': {'semeval-2016-2017-task3-subtaskBC': {'num_records': -1, 'record_format': 'dict', 'file_size': 6344358, 'reader_code': 'https://github.com/RaRe-Technologies/gensim-data/releases/download/semeval-2016-2017-task3-subtaskB-eng/__init__.py', 'license': 'All files released for the task are free for general research use', 'fields': {'2016-train': ['...'], '2016-dev': ['...'], '2017-test': ['...'], '2016-test': ['...']}, 'description': 'SemEval 2016 / 2017 Task 3 Subtask B and C datasets contain train+development (317 original questions, 3,169 related questions, and 31,690 comments), and test datasets in English. The description of the tasks and the collected data is given in sections 3 and 4.1 of the task paper http://alt.qcri.org/semeval2016/task3/data/uploads/semeval2016-task3-report.pdf linked in section “Papers” of https://github.com/RaRe-Technologies/gensim-data/issues/18.', 'checksum': '701ea67acd82e75f95e1d8e62fb0ad29', 'file_name': 'semeval-2016-2017-task3-subtaskBC.gz',

In [ ]:
# Датасет 1
w2v_model_2 = api.load('word2vec-ruscorpora-300')

print(f"Размер словаря: {len(w2v_model_2.key_to_index)}")
print(f"Размерность векторов: {w2v_model_2.vector_size}")

[==================================================] 100.0% 198.8/198.8MB downloaded
Размер словаря: 184973
Размерность векторов: 300


**Источник данных**: Русский национальный корпус

**Объем**: 184 973 слова, размер словаря: 198.8 МБ, размерность векторов: 300

**Назначение**: датасет представляет собой предобученные векторные представления слов (word embeddings) для русского языка, полученные с помощью модели word2vec. Используется для:

*   Анализа семантической близости русских слов
*   Классификации текстов на русском языке
*   Машинного перевода и информационного поиска
*   Сентимент-анализа русскоязычных текстов

In [ ]:
# Датасет 2
w2v_model_3 = api.load('word2vec-google-news-300')

print(f"Размер словаря: {len(w2v_model_3.key_to_index)}")
print(f"Размерность векторов: {w2v_model_3.vector_size}")

[==================================================] 100.0% 1662.8/1662.8MB downloaded
Размер словаря: 3000000
Размерность векторов: 300


**Источник данных**: новостной датасет Google News

**Объем**: 3 миллиона слов и фраз, размер словаря: 1.62 ГБ (1662.8 MB), размерность векторов: 300

**Назначение**: Это одна из самых известных предобученных моделей word2vec, которая включает не только отдельные слова, но и словосочетания. Используется для:
*   Изучения семантических и синтаксических закономерностей в английском языке
*   Решения задач аналогий (word analogies), например, задача "король - мужчина + женщина = королева"
*   Кластеризации документов и тематического моделирования
*   Улучшения качества NLP-моделей за счет готовых эмбеддингов

In [ ]:
# Датасет 3
w2v_model_4 = api.load('glove-twitter-25')

print(f"Размер словаря: {len(w2v_model_4.key_to_index)}")
print(f"Размерность векторов: {w2v_model_4.vector_size}")

[==================================================] 100.0% 104.8/104.8MB downloaded
Размер словаря: 1193514
Размерность векторов: 25


**Источник данных**: корпус из 2 миллиардов твитов (27 миллиардов токенов, 1.2 миллиона уникальных слов, все слова в нижнем регистре).

**Объем**: 1 193 515 слов, размер словаря: 104.8 МБ, размерность векторов: 25.

**Назначение**: это предобученная модель векторных представлений слов (word embeddings), созданная с помощью алгоритма GloVE (Global Vectors). Датасет обучен на данных из социальной сети Twitter (X). Используется для:

*   Анализа тональности (сентимент-анализа) твитов и комментариев в социальных сетях, где много сленга, эмодзи и нестандартного написания слов
*   Классификации коротких текстов, таких как сообщения в чатах, отзывы или поисковые запросы
*   Решения задач на аналогии в неформальной лексике, например, нахождение связей между эмодзи и словом вроде " 😂" и "смешно"
*   Изучения семантики специфических для соцсетей терминов, которые редко встречаются в новостях или Википедии

**Важная особенность**: Из-за специфики тренировочных данных (твиты), модель может не содержать векторов для некоторых стандартных слов, например, для цифр ("1", "2"...) . Зато в ней можно найти представления для эмодзи и сленговых выражений.

**Базовые операции с векторами**

In [ ]:
# Получаем вектор слова
vector = w2v_model_1['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model_1.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [ ]:
# Находим похожие слова
similar_words = w2v_model_1.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'python':
  monty: 0.6886
  php: 0.5865
  perl: 0.5784
  cleese: 0.5447
  flipper: 0.5113


*Ваш ответ здесь*

**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [ ]:
w2v_model_4 = api.load('glove-twitter-25')

print(f"Размер словаря: {len(w2v_model_4.key_to_index)}")
print(f"Размерность векторов: {w2v_model_4.vector_size}")

Размер словаря: 1193514
Размерность векторов: 25


2. Напишите функцию, которая принимает на вход любое слово и возвращает 10 наиболее близких по вектору слов

In [ ]:
def find_similar_words(word, model, topn=10):

    # Проверяем, есть ли слово в словаре модели
    if word not in w2v_model_4:
        print(f"Слово 'funny' не найдено в словаре модели")
        return []

    # Получаем вектор слова
    vector = w2v_model_4['funny']
    print(f"Вектор слова 'funny': {vector[:5]}...")  # Показываем первые 5 чисел

    # Находим похожие слова
    similar_words = w2v_model_4.most_similar('funny', topn=10)

    return similar_words

result = find_similar_words('funny', w2v_model_4, topn=10)

print("Слова, похожие на 'funny':")
for word, score in result:
  print(f"  {word}: {score:.4f}")

Вектор слова 'funny': [ 0.42466 -0.23493  0.67394 -0.51295  0.67063]...
Слова, похожие на 'funny':
  weird: 0.9369
  hilarious: 0.9361
  thats: 0.9352
  crazy: 0.9340
  stupid: 0.9316
  kinda: 0.9284
  silly: 0.9272
  confused: 0.9236
  dude: 0.9215
  dumb: 0.9198


3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [ ]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [ ]:
skipgram_model = Word2Vec(
    sentences=cooking_sentences,
    vector_size=50,       # размерность векторов
    window=3,             # размер контекстного окна
    min_count=1,          # минимальная частота слова
    workers=2,            # количество ядер
    sg=1                  # 1 = Skip-Gram
)

print(f"Размер словаря: {len(skipgram_model.wv.key_to_index)}")

Размер словаря: 65


In [ ]:
print(f"Слова в словаре: {list(w2v_model_4.key_to_index.keys())[:10]}...")

Слова в словаре: ['<user>', '.', ':', 'rt', ',', '<repeat>', '<hashtag>', '<number>', '<url>', '!']...


4. Проверьте модель

In [ ]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = w2v_model_4.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':
  борщ: 0.9473
  шить: 0.9186
  жарить: 0.9179
  готовить: 0.9172
  челку: 0.9095


In [ ]:
# Найдите слова, похожие на "духовка"
try:
    similar = w2v_model_4.most_similar('духовка', topn=5)
    print("Слова, похожие на 'духовка':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'духовка' не найдено в словаре")

# Найдите слова, похожие на "овощи"
try:
    similar = w2v_model_4.most_similar('овощи', topn=5)
    print("Слова, похожие на 'овощи':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'овощи' не найдено в словаре")

Слово 'духовка' не найдено в словаре
Слова, похожие на 'овощи':
  фрукты: 0.9272
  столы: 0.8663
  курицу: 0.8606
  носки: 0.8594
  зеленые: 0.8577


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [ ]:
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [ ]:
# Слова, похожие на "варить"
try:
    similar = ft_model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

# Слова, похожие на "духовка"
try:
    similar = ft_model.wv.most_similar('духовка', topn=5)
    print("Слова, похожие на 'духовка':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'духовка' не найдено в словаре")

# Слова, похожие на "овощи"
try:
    similar = ft_model.wv.most_similar('овощи', topn=5)
    print("Слова, похожие на 'овощи':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'овощи' не найдено в словаре")

Слова, похожие на 'варить':
  жарить: 0.5353
  парить: 0.4805
  месить: 0.3541
  тушить: 0.3405
  специи: 0.2622
Слова, похожие на 'духовка':
  взбивать: 0.4565
  лимон: 0.3561
  салат: 0.3050
  курица: 0.3041
  тост: 0.2944
Слова, похожие на 'овощи':
  жарить: 0.2960
  фольга: 0.2574
  морковь: 0.2297
  соус: 0.2172
  торт: 0.2094


7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [ ]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = w2v_model_4.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для разных слов
compare_models('learning')
compare_models('neural')

compare_models('traiin')
compare_models('wendow')
compare_models('butiful')
# Word2Vec показывает более худшие результаты


Сравнение для слова: 'learning'
  Word2Vec: ['creating', 'lessons']
  FastText: ['духовка', 'пирог']

Сравнение для слова: 'neural'
  Word2Vec: ['peripheral', 'hand-held']
  FastText: ['мука', 'травы']

Сравнение для слова: 'traiin'
  Word2Vec: слово не найдено
  FastText: ['суп', 'барбекю']

Сравнение для слова: 'wendow'
  Word2Vec: слово не найдено
  FastText: ['бекон', 'масло']

Сравнение для слова: 'butiful'
  Word2Vec: ['btfl', 'strng']
  FastText: ['курица', 'лимон']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [ ]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [ ]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [ ]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [ ]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


8. Сравните схожесть doc_2 и doc_4

In [ ]:
# Сравниваем схожесть документов
def compare_documents(doc2_id, doc4_id):
    similarity = doc_model.dv.similarity(f"doc_{doc2_id}", f"doc_{doc4_id}")
    print(f"Схожесть doc_{doc2_id} и doc_{doc4_id}: {similarity:.4f}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")
    print(f"  doc_{doc4_id}: {documents[doc4_id]}")

compare_documents(2, 4)  # machine learning vs deep learning

Схожесть doc_2 и doc_4: -0.0362
  doc_2: python programming for data science
  doc_4: computer vision processes images


9. Найдите самый похожий документ на doc_1

In [ ]:
# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_1", topn=2)
print("\nДокументы, похожие на doc_1:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")


Документы, похожие на doc_1:
  doc_0: 0.2735
    Текст: machine learning is interesting
  doc_3: 0.2031
    Текст: artificial intelligence is amazing


10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [ ]:
# Создадим синтетический датасет для тестирования модели Doc2Vec на основе модели w2v_model_4
synthetic_sentences = [
    ['run', 'race', 'fast', 'win', 'competition'],
    ['cat', 'dog', 'pet', 'animal', 'house'],
    ['food', 'eat', 'restaurant', 'menu', 'dinner'],
    ['car', 'drive', 'road', 'fast', 'destination'],
    ['computer', 'program', 'code', 'software', 'developer'],
    ['travel', 'trip', 'hotel', 'flight', 'vacation'],
    ['sport', 'game', 'play', 'team', 'ball'],
    ['python', 'language', 'script', 'data', 'algorithm'],
]

print(f"\nСоздано {len(synthetic_sentences)} синтетических предложений")


Создано 8 синтетических предложений


In [ ]:
# Преобразуем в формат TaggedDocument
tagged_documents = []
for i, sentence in enumerate(synthetic_sentences):
    tagged_document = TaggedDocument(words=sentence, tags=[f"doc_{i}"])
    tagged_documents.append(tagged_document)

print("Размеченные документы:")
for doc in tagged_documents:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")
    print()

Размеченные документы:
  Слова: ['run', 'race', 'fast', 'win', 'competition']
  Тег: ['doc_0']

  Слова: ['cat', 'dog', 'pet', 'animal', 'house']
  Тег: ['doc_1']

  Слова: ['food', 'eat', 'restaurant', 'menu', 'dinner']
  Тег: ['doc_2']

  Слова: ['car', 'drive', 'road', 'fast', 'destination']
  Тег: ['doc_3']

  Слова: ['computer', 'program', 'code', 'software', 'developer']
  Тег: ['doc_4']

  Слова: ['travel', 'trip', 'hotel', 'flight', 'vacation']
  Тег: ['doc_5']

  Слова: ['sport', 'game', 'play', 'team', 'ball']
  Тег: ['doc_6']

  Слова: ['python', 'language', 'script', 'data', 'algorithm']
  Тег: ['doc_7']



Обучаем Doc2Vec с векторной размерностью 10

In [ ]:
doc_model = Doc2Vec(
    documents=tagged_documents,
    vector_size=10,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 8


In [ ]:
# Найдем похожие документы на doc_0
# ['run', 'race', 'fast', 'win', 'competition']
similar_documents = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_documents:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {tagged_documents[doc_id]}")

# Должно быть похоже на doc_6 по теме спорт


Документы, похожие на doc_0:
  doc_2: 0.3788
    Текст: TaggedDocument<['food', 'eat', 'restaurant', 'menu', 'dinner'], ['doc_2']>
  doc_7: 0.2726
    Текст: TaggedDocument<['python', 'language', 'script', 'data', 'algorithm'], ['doc_7']>


In [ ]:
# Найдем похожие документы на doc_4
# ['computer', 'program', 'code', 'software', 'developer']
similar_documents = doc_model.dv.most_similar("doc_4", topn=2)
print("\nДокументы, похожие на doc_4:")
for doc_tag, similarity in similar_documents:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {tagged_documents[doc_id]}")

# Должно быть похоже на doc_7 по теме компьютер


Документы, похожие на doc_4:
  doc_2: 0.3216
    Текст: TaggedDocument<['food', 'eat', 'restaurant', 'menu', 'dinner'], ['doc_2']>
  doc_6: 0.1729
    Текст: TaggedDocument<['sport', 'game', 'play', 'team', 'ball'], ['doc_6']>


In [ ]:
# Найдем похожие документы на doc_5
# ['travel', 'trip', 'hotel', 'flight', 'vacation']
similar_documents = doc_model.dv.most_similar("doc_5", topn=2)
print("\nДокументы, похожие на doc_5:")
for doc_tag, similarity in similar_documents:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {tagged_documents[doc_id]}")

# Тема путешествия совпадает с doc_3


Документы, похожие на doc_5:
  doc_7: 0.4579
    Текст: TaggedDocument<['python', 'language', 'script', 'data', 'algorithm'], ['doc_7']>
  doc_3: 0.3219
    Текст: TaggedDocument<['car', 'drive', 'road', 'fast', 'destination'], ['doc_3']>


Обучаем Doc2Vec с векторной размерностью 50

In [ ]:
doc_model = Doc2Vec(
    documents=tagged_documents,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 8


In [ ]:
# Найдем похожие документы на doc_0
# ['run', 'race', 'fast', 'win', 'competition']
similar_documents = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_documents:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {tagged_documents[doc_id]}")

# Должно быть похоже на doc_6 по теме спорт
# Частично совпадает с doc_5


Документы, похожие на doc_0:
  doc_5: 0.2768
    Текст: TaggedDocument<['travel', 'trip', 'hotel', 'flight', 'vacation'], ['doc_5']>
  doc_1: 0.2747
    Текст: TaggedDocument<['cat', 'dog', 'pet', 'animal', 'house'], ['doc_1']>


In [ ]:
# Найдем похожие документы на doc_4
# ['computer', 'program', 'code', 'software', 'developer']
similar_documents = doc_model.dv.most_similar("doc_4", topn=2)
print("\nДокументы, похожие на doc_4:")
for doc_tag, similarity in similar_documents:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {tagged_documents[doc_id]}")

# Должно быть похоже на doc_7 по теме компьютер
# Показывает отрицательное значение к doc_7


Документы, похожие на doc_4:
  doc_7: -0.0185
    Текст: TaggedDocument<['python', 'language', 'script', 'data', 'algorithm'], ['doc_7']>
  doc_5: -0.0311
    Текст: TaggedDocument<['travel', 'trip', 'hotel', 'flight', 'vacation'], ['doc_5']>


In [ ]:
# Найдем похожие документы на doc_5
# ['travel', 'trip', 'hotel', 'flight', 'vacation']
similar_documents = doc_model.dv.most_similar("doc_5", topn=2)
print("\nДокументы, похожие на doc_5:")
for doc_tag, similarity in similar_documents:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {tagged_documents[doc_id]}")

# Должно быть похоже на doc_3 по теме путешествия


Документы, похожие на doc_5:
  doc_0: 0.2768
    Текст: TaggedDocument<['run', 'race', 'fast', 'win', 'competition'], ['doc_0']>
  doc_1: 0.2490
    Текст: TaggedDocument<['cat', 'dog', 'pet', 'animal', 'house'], ['doc_1']>


Обучаем Doc2Vec с векторной размерностью 100

In [ ]:
doc_model = Doc2Vec(
    documents=tagged_documents,
    vector_size=100,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 8


In [ ]:
# Найдем похожие документы на doc_0
# ['run', 'race', 'fast', 'win', 'competition']
similar_documents = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_documents:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {tagged_documents[doc_id]}")

# Должно быть похоже на doc_6 по теме спорт


Документы, похожие на doc_0:
  doc_1: 0.1677
    Текст: TaggedDocument<['cat', 'dog', 'pet', 'animal', 'house'], ['doc_1']>
  doc_7: 0.1466
    Текст: TaggedDocument<['python', 'language', 'script', 'data', 'algorithm'], ['doc_7']>


In [ ]:
# Найдем похожие документы на doc_4
# ['computer', 'program', 'code', 'software', 'developer']
similar_documents = doc_model.dv.most_similar("doc_4", topn=2)
print("\nДокументы, похожие на doc_4:")
for doc_tag, similarity in similar_documents:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {tagged_documents[doc_id]}")

# Должно быть похоже на doc_7 по теме компьютер


Документы, похожие на doc_4:
  doc_6: 0.0830
    Текст: TaggedDocument<['sport', 'game', 'play', 'team', 'ball'], ['doc_6']>
  doc_5: 0.0713
    Текст: TaggedDocument<['travel', 'trip', 'hotel', 'flight', 'vacation'], ['doc_5']>


In [ ]:
# Найдем похожие документы на doc_5
# ['travel', 'trip', 'hotel', 'flight', 'vacation']
similar_documents = doc_model.dv.most_similar("doc_5", topn=2)
print("\nДокументы, похожие на doc_5:")
for doc_tag, similarity in similar_documents:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {tagged_documents[doc_id]}")

# Должно быть похоже на doc_3 по теме путешествия


Документы, похожие на doc_5:
  doc_0: 0.1395
    Текст: TaggedDocument<['run', 'race', 'fast', 'win', 'competition'], ['doc_0']>
  doc_1: 0.1132
    Текст: TaggedDocument<['cat', 'dog', 'pet', 'animal', 'house'], ['doc_1']>


###Результаты обучения модели Doc2Vec

**doc_0 ['run', 'race', 'fast', 'win', 'competition']**
1. Векторная размерность: 10
*   doc_2: 0.3788 ['food', 'eat', 'restaurant', 'menu', 'dinner']
*   doc_7: 0.2726 ['python', 'language', 'script', 'data', 'algorithm']
2. Векторная размерность: 50
*   doc_5: 0.2768 ['travel', 'trip', 'hotel', 'flight', 'vacation']
*   doc_1: 0.2747 ['cat', 'dog', 'pet', 'animal', 'house']
3. Векторная размерность: 100
*   doc_1: 0.1677 ['cat', 'dog', 'pet', 'animal', 'house']
*   doc_7: 0.1466 ['python', 'language', 'script', 'data', 'algorithm']

**doc_4 ['computer', 'program', 'code', 'software', 'developer']**
1. Векторная размерность: 10
*   doc_2: 0.3216 ['food', 'eat', 'restaurant', 'menu', 'dinner']
*   doc_6: 0.1729 ['sport', 'game', 'play', 'team', 'ball']
2. Векторная размерность: 50
*   doc_7: -0.0185 ['python', 'language', 'script', 'data', 'algorithm']
*   doc_5: -0.0311 ['travel', 'trip', 'hotel', 'flight', 'vacation']
3. Векторная размерность: 100
*   doc_6: 0.0830 ['sport', 'game', 'play', 'team', 'ball']
*   doc_5: 0.0713 ['travel', 'trip', 'hotel', 'flight', 'vacation']

**doc_5 ['travel', 'trip', 'hotel', 'flight', 'vacation']**
1. Векторная размерность: 10
*   doc_7: 0.4579 ['python', 'language', 'script', 'data', 'algorithm']
*   doc_3: 0.3219 ['car', 'drive', 'road', 'fast', 'destination']
2. Векторная размерность: 50
*   doc_0: 0.2768 ['run', 'race', 'fast', 'win', 'competition']
*   doc_1: 0.2490 ['cat', 'dog', 'pet', 'animal', 'house']
3. Векторная размерность: 100
*   doc_0: 0.1395 ['run', 'race', 'fast', 'win', 'competition']
*   doc_1: 0.1132 ['cat', 'dog', 'pet', 'animal', 'house']

###Выводы
Модель довольно плохо справляется с поиском семантически похожих документов при разных векторных размерностях. При векторной размерности 50 модель только один раз показала отрицательную схожесть, однако это плохой результат, потому что doc_4 и doc_7 должны иметь положительную косинусную схожесть. Модель справилась только в случае с doc_5 при векторной размерности 10 (doc_3). Чаще всего модель дублирует результат при разных векторных размерностях.